# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omesh-kumar/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

The paper reports that refreshing existing content can improve search performance.

**Methodology Question:**
How were the pages selected for refresh? Were they randomly selected or chosen because they already had strong potential? This helps determine whether selection bias may have influenced the results.

---

## Finding 2

The paper suggests that content opportunity scores are associated with future content performance.

**Methodology Question:**
Was the validation performed on unseen clients or future time periods? This would provide stronger evidence that the findings generalize beyond the observed dataset.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [29]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

# Baseline score
df["baseline_score"] = (
    (df["days_since_last_update"] / 30)
    + (df["ctr"] * 100)
    + ((df["impressions_90d"] / 1000))
)

# Remove leakage columns
leakage_cols = [
    "baseline_score",
    "ctr",
    "impressions_90d",
    "days_since_last_update"
]

X = df.select_dtypes(include=np.number).drop(columns=leakage_cols)
y = df["baseline_score"]

In [30]:
!git clone https://github.com/Omesh-kumar/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [31]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [32]:
!ls data/raw

content_refresh_anonymized.csv


In [33]:
!ls /content

flyrank-ml-internship  sample_data


In [34]:
!ls /content/flyrank-ml-internship

AGENTS.md			   GUIDE.md	     SETUP.md
CLAUDE.md			   LICENSE	     skills
Copy_of_w02_ml_task_framing.ipynb  notebooks	     submission
data				   outputs	     w02_ml_task_mapping.ipynb
DATA_USE.md			   README.md	     work
docs				   requirements.txt
flyrank-ml-internship		   scripts


In [36]:
# BEFORE: Random Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

before_r2 = r2_score(y_test, pred)
before_mae = mean_absolute_error(y_test, pred)

print("Random Split")
print("R2 :", round(before_r2,3))
print("MAE:", round(before_mae,3))

Random Split
R2 : 0.882
MAE: 12.222


In [37]:
# AFTER: Grouped Split

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

after_r2 = r2_score(y_test, pred)
after_mae = mean_absolute_error(y_test, pred)

print("Grouped Split")
print("R2 :", round(after_r2,3))
print("MAE:", round(after_mae,3))

Grouped Split
R2 : 0.698
MAE: 8.66


In [38]:
comparison = pd.DataFrame({
    "Validation": ["Random Split", "Grouped Split"],
    "MAE": [round(before_mae,3), round(after_mae,3)],
    "R2": [round(before_r2,3), round(after_r2,3)]
})

comparison

,Validation,MAE,R2
0,Random Split,12.222,0.882
1,Grouped Split,8.660,0.698


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage Audit

I reviewed the final feature set for possible data leakage.

- No client identifier was used as a model feature.
- No target or future information was included in the feature set.
- Features are based on historical observations available before prediction.
- Validation was performed using GroupShuffleSplit to reduce leakage between clients.

Conclusion:
I did not find evidence of direct data leakage in the final feature set.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim Rewrite

Observed result:
Using GroupShuffleSplit reduced the reported R² compared with a random split.

Measured evidence:
The validation results changed from R² = 0.882 (Random Split) to R² = 0.698 (Grouped Split).

Directional conclusion:
This suggests that random splitting may produce more optimistic estimates than grouped validation.

Decision-support statement:
Grouped validation provides a more reliable estimate of expected model performance on unseen clients.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.